In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from models import RawPosting, FilteredPosting, SourceType, AgeClassification, StaleSignal

# 1. Simular la salida del Agente de Scraping (RawPosting)
try:
    raw_data = RawPosting(
        title="Senior Data Engineer",
        department="Data & Analytics",
        location="San José, Costa Rica",
        posted_date="2026-05-25",
        job_id="12345",
        url="https://example.com/job/12345",
        description_snippet="Looking for a Principal Engineer with Databricks experience...",
        source_type=SourceType.LINKEDIN
    )
    print("✅ RawPosting validado correctamente.")
except Exception as e:
    print(f"❌ Error de validación: {e}")

# 2. Simular la transformación del Agente de Filtrado (FilteredPosting)
# Si intentas pasar un string inválido en stale_signal, Pydantic fallará en tiempo de ejecución.
filtered_data = FilteredPosting(
    posting=raw_data,
    age_classification=AgeClassification.ACTIVE,
    stale_signal=StaleSignal.NONE,
    archive_flag=False
)
print(f"✅ Pipeline listo. Destino de archivo: {filtered_data.archive_flag}")

✅ RawPosting validado correctamente.
✅ Pipeline listo. Destino de archivo: False


In [4]:
from pydantic import ValidationError

try:
    # Si el agente olvida la ubicación, Pydantic frena el sistema aquí
    posting_invalido = RawPosting(
        title="Data Engineer",
        url="https://example.com",
        description_snippet="..."
        # Falta 'location' y 'source_type'
    )
except ValidationError as e:
    print(f"El Agente falló el contrato: {e}")
    # Aquí puedes programar un re-intento (retry) o una alerta

El Agente falló el contrato: 2 validation errors for RawPosting
location
  Field required [type=missing, input_value={'title': 'Data Engineer'...ription_snippet': '...'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
source_type
  Field required [type=missing, input_value={'title': 'Data Engineer'...ription_snippet': '...'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [5]:
from models import RawPosting, FilteredPosting, SourceType, AgeClassification, StaleSignal
from pydantic import ValidationError

# Datos de prueba simulando que traen el campo inesperado 'provincia'
datos_con_provincia = {
    "title": "Senior Data Engineer",
    "department": "Data & Analytics",
    "location": "San José",
    "provincia": "San José",  # <-- Campo extra de internet
    "posted_date": "2026-05-25",
    "job_id": "12345",
    "url": "https://example.com/job/12345",
    "description_snippet": "Looking for a Principal Engineer...",
    "source_type": SourceType.LINKEDIN
}

print("--- 1. Probando RawPosting (Debe permitir 'provincia') ---")
try:
    raw_data = RawPosting(**datos_con_provincia)
    print("✅ RawPosting creado con éxito.")
    print(f"Campos guardados oficialmente: {list(raw_data.model_fields.keys())}")
    print(f"Campos extra capturados dinámicamente: {raw_data.model_extra}")
    print(f"Acceso directo al extra: raw_data.provincia -> '{raw_data.provincia}'")
except Exception as e:
    print(f"❌ Falló inesperadamente: {e}")

print("\n--- 2. Probando FilteredPosting (Debe prohibir campos extra) ---")
try:
    # Intentamos inyectar un campo inventado por el LLM en la raíz del Filtro
    filtered_data = FilteredPosting(
        posting=raw_data,
        age_classification=AgeClassification.ACTIVE,
        stale_signal=StaleSignal.NONE,
        archive_flag=False,
        campo_alucinado_por_ia="un_valor_cualquiera"  # <-- Esto debe disparar el error
    )
except ValidationError as e:
    print("✅ Pydantic detuvo la alucinación correctamente en la frontera del Agente.")
    print(f"Error esperado capturado:\n{e}")

--- 1. Probando RawPosting (Debe permitir 'provincia') ---
✅ RawPosting creado con éxito.
Campos guardados oficialmente: ['title', 'department', 'location', 'posted_date', 'job_id', 'url', 'description_snippet', 'source_type']
Campos extra capturados dinámicamente: {'provincia': 'San José'}
Acceso directo al extra: raw_data.provincia -> 'San José'

--- 2. Probando FilteredPosting (Debe prohibir campos extra) ---
✅ Pydantic detuvo la alucinación correctamente en la frontera del Agente.
Error esperado capturado:
1 validation error for FilteredPosting
campo_alucinado_por_ia
  Extra inputs are not permitted [type=extra_forbidden, input_value='un_valor_cualquiera', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


/var/folders/h6/tzdh3vtn0wn1sbmjc4bbtfpr0000gn/T/ipykernel_7775/2838637234.py:21: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print(f"Campos guardados oficialmente: {list(raw_data.model_fields.keys())}")


In [6]:
from models import RawPosting, FilteredPosting, SourceType, AgeClassification, StaleSignal
from pydantic import ValidationError

# Datos de prueba simulando que traen el campo inesperado 'provincia'
datos_con_provincia = {
    "title": "Senior Data Engineer",
    "department": "Data & Analytics",
    "location": "San José",
    "provincia": "San José",  # <-- Campo extra de internet
    "posted_date": "2026-05-25",
    "job_id": "12345",
    "url": "https://example.com/job/12345",
    "description_snippet": "Looking for a Principal Engineer...",
    "source_type": SourceType.LINKEDIN
}

print("--- 1. Probando RawPosting (Debe permitir 'provincia') ---")
try:
    raw_data = RawPosting(**datos_con_provincia)
    print("✅ RawPosting creado con éxito.")
    
    # CORRECCIÓN V2.11: Acceder a model_fields desde la Clase, no desde la instancia
    print(f"Campos guardados oficialmente: {list(RawPosting.model_fields.keys())}")
    
    print(f"Campos extra capturados dinámicamente: {raw_data.model_extra}")
    
    # CORRECCIÓN DE ACCESO: Los extras se buscan en el diccionario .model_extra
    if raw_data.model_extra and "provincia" in raw_data.model_extra:
        print(f"✅ Acceso correcto al extra: raw_data.model_extra['provincia'] -> '{raw_data.model_extra['provincia']}'")
        
except Exception as e:
    print(f"❌ Falló inesperadamente: {e}")

print("\n--- 2. Probando FilteredPosting (Debe prohibir campos extra) ---")
try:
    # Intentamos inyectar un campo inventado por el LLM en la raíz del Filtro
    filtered_data = FilteredPosting(
        posting=raw_data,
        age_classification=AgeClassification.ACTIVE,
        stale_signal=StaleSignal.NONE,
        archive_flag=False,
        campo_alucinado_por_ia="un_valor_cualquiera"  # <-- Esto debe disparar el error
    )
except ValidationError as e:
    print("✅ Pydantic detuvo la alucinación correctamente en la frontera del Agente.")
    print(f"Error esperado capturado:\n{e}")

--- 1. Probando RawPosting (Debe permitir 'provincia') ---
✅ RawPosting creado con éxito.
Campos guardados oficialmente: ['title', 'department', 'location', 'posted_date', 'job_id', 'url', 'description_snippet', 'source_type']
Campos extra capturados dinámicamente: {'provincia': 'San José'}
✅ Acceso correcto al extra: raw_data.model_extra['provincia'] -> 'San José'

--- 2. Probando FilteredPosting (Debe prohibir campos extra) ---
✅ Pydantic detuvo la alucinación correctamente en la frontera del Agente.
Error esperado capturado:
1 validation error for FilteredPosting
campo_alucinado_por_ia
  Extra inputs are not permitted [type=extra_forbidden, input_value='un_valor_cualquiera', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
